# 12. Lecke - Csevegési Előzmények Csökkentése Agent Scratchpaddel

Ez a jegyzetfüzet bemutatja, hogyan kezeljük a kontextust hosszú beszélgetések során a Microsoft Agent Framework segítségével. Ahogy a beszélgetések nőnek, a tokenek száma is növekszik — végül meghaladva a modell kontextusablakát. Ezt egy **kontextus összefoglaló mintával** és egy **agent scratchpaddel** oldjuk meg a tartós memória érdekében.

## Amit Megtanulsz:
1. **Miért Fontos a Kontextus Kezelése**: Token korlátok és kontextusablakok megértése
2. **Kontextus-Tudatos Ügynökök**: Ügynökök építése, amelyek kezelik a saját beszélgetési kontextusukat
3. **Kontextus Összefoglaló Minta**: Eszközök használata a beszélgetési előzmények tömörítésére
4. **Agent Scratchpad**: Tartós memória, amely túléli a kontextus csökkentést

## Előfeltételek:
- Azure OpenAI beállítása környezeti változókkal konfigurálva
- Az alapvető ügynök koncepciók megértése az előző leckékből


## Beállítás


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Miért fontos a kontextuskezelés

Minden LLM-nek van egy véges **kontextusablaka** — a maximális token szám, amit egyetlen kérésben feldolgozni tud. Amint egy többkörös beszélgetés halad:

- A **tokenek száma lineárisan növekszik** minden felhasználói üzenettel és asszisztensi válasszal.
- A **prompt tokenek dominálják a költséget**, mert a teljes előzmény minden körben újraküldésre kerül.
- Végül a beszélgetés **túllépi a kontextusablakot**, és a modell vagy levágja, vagy hibát jelez.

### Kontextuskezelési stratégiák

| Stratégia | Hogyan működik | Kereskedés |
|---|---|---|
| **Levágás** | A legrégebbi üzenetek törlése | Korai kontextus elveszik |
| **Összefoglalás** | Régebbi üzenetek összesűrítése egy összefoglalóba | Néhány részlet elveszik, de a kulcspontok megmaradnak |
| **Jegyzetblok / Külső memória** | Kulcsfontosságú tények tárolása a beszélgetésen kívül | Eszközhívásokat igényel, de túléli a bármilyen csökkentést |

Ebben a jegyzetfüzetben az **összefoglalást** egy **jegyzeteszközzel** kombináljuk, így az ügynök fenntarthatja a folytonosságot akkor is, ha a beszélgetési előzmény tömörítésre kerül.


## Kontextusérzékeny ügynök létrehozása


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Egy hosszú beszélgetés szimulálása

Nézzünk meg egy többfordulós beszélgetést, hogy lássuk, hogyan halmozódik a kontextus. Az ügynöknek meg kell őriznie a kulcsfontosságú részleteket (preferenciák, költségvetés, utazási dátumok) a fordulók között, és folytonosságot kell mutatnia.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Figyeld meg, hogyan őrzi az ügynök a korábbi körök kontextusát — tud Japánról, a sushiról, a templomokról, a fotózásról, a 3000 dolláros költségvetésről, az egyéni utazásról és az áprilisi időszakról. Egy rövid beszélgetésnél ez jól működik, de ahogy a beszélgetés nő, a teljes előzményt újraküldeni költséges lesz.

Folytassuk a beszélgetést több körrel, hogy lássuk a kontextus felhalmozódását:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Kontextus összefoglaló minta

Ahogy a beszélgetés nő, használhatunk egy **összefoglaló eszközt**, hogy a felhalmozott kontextust tömör formába sűrítsük. Az ügynök ezt az eszközt hívja meg, hogy rögzítse a kulcsfontosságú preferenciákat, így még ha a régebbi üzeneteket el is távolítják, a lényeges információ megmarad.

Ez a minta az összetettebb előzménytömörítés alapköve:
1. Az ügynök azonosítja a beszélgetés kulcsfontosságú tényeket
2. Meghívja az összefoglaló eszközt, hogy eltárolja ezeket
3. A régebbi üzenetek biztonságosan eltávolíthatók, mert az összefoglaló tartalmazza a fontosakat

Alább definiálunk egy `summarize_preferences` eszközt, amelyet az ügynök használhat a megtanultak tömör összefoglalójának rögzítésére.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Összefoglaló

Ebben a leckében megtanultad, hogyan kezeld a kontextust hosszú ideig zajló ügynökbeszélgetések során a Microsoft Agent Framework használatával:

### Kulcsfogalmak
- **A kontextusablakok végesek** — a beszélgetés előzményeiben szereplő minden token pénzbe kerül és beleszámít a korlátba.
- **Összegző eszközök** lehetővé teszik az ügynök számára, hogy a felhalmozott kontextust tömör összefoglalókba sűrítse, csökkentve a tokenhasználatot miközben megőrzi a lényegi információkat.
- **Ügynök jegyzetfüzetek** tartós külső memóriát biztosítanak, amelyek túlélnek bármilyen beszélgetés-csökkentést.

### Amit Építettél
- Egy **kontextusérzékeny ügynököt**, amely fenntartja a folytonosságot többszörös fordulós beszélgetések során
- Egy **összegző eszközt** (`summarize_preferences`), amely kulcsfontosságú felhasználói adatokat rögzít tömör formában
- Egy **többszörös fordulós beszélgetést**, amely bemutatja a kontextus megtartását és a változások kezelését

### Valós Alkalmazások
- **Ügyfélszolgálati botok**: Emlékezzenek a preferenciákra a hosszú támogatási munkamenetek alatt
- **Személyi asszisztensek**: Kövessék a folyamatban lévő projekteket anélkül, hogy újra magyarázni kellene a kontextust
- **Oktató tutorok**: Fenntartsák a tanulói előrehaladást több interakció során

### Következő lépések
- Valósíts meg egy teljes jegyzetfüzet eszközt fájlalapú tartóssággal
- Adj hozzá automatikus előzménycsökkentést az összegzés után
- Kombináld vektoradatbázisokkal szemantikai memória kereséshez
- Építs ügynököket, amelyek napokkal később folytatni tudják a beszélgetéseket teljes kontextussal


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
